# Visual Token Compression for Efficient VLM Inference
### HPML Course Project - LLaVA-1.5-7B + VQAv2

**Sections:**
- Section 0: Environment Setup
- Section 1: Dataset Preparation
- Section 2: Model Loading + Hook Injection
- Section 3: Compression Strategies (Uniform / Attention / Merge / Pool)
- Section 4: Benchmark Runner
- Section 5: Run All Experiments
- Section 6: Results & Visualization
- Section 7: Fine-grained Analysis (Yes/No vs Number vs Open-ended)
- Section 8: Interactive Demo

## Section 0 - Environment Setup

In [ ]:
!pip install -q transformers==4.40.0 accelerate bitsandbytes sentencepiece
!pip install -q Pillow matplotlib pandas seaborn datasets einops timm
!pip install -q git+https://github.com/haotian-liu/LLaVA.git

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/HPML_Project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Results will be saved to: {SAVE_DIR}')

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Section 1 - Dataset Preparation
Loading 1000 samples from VQAv2 validation split.
We preserve `answer_type` labels for fine-grained analysis in Section 7.

In [ ]:
from datasets import load_dataset
import pandas as pd
from PIL import Image

print('Loading VQAv2 validation split (1000 samples)...')
raw_ds = load_dataset('HuggingFace/VQAv2', split='validation[:1000]', trust_remote_code=True)

samples = []
for item in raw_ds:
    answers = item['answers']
    answer_counts = {}
    for a in answers:
        ans = a['answer'].strip().lower()
        answer_counts[ans] = answer_counts.get(ans, 0) + 1
    gt_answer = max(answer_counts, key=answer_counts.get)
    samples.append({
        'question_id': item['question_id'],
        'question':    item['question'],
        'gt_answer':   gt_answer,
        'answer_type': item['answer_type'],
        'image':       item['image'],
    })

print(f'Loaded {len(samples)} samples')
print('Answer type distribution:')
print(pd.Series([s['answer_type'] for s in samples]).value_counts())

## Section 2 - Model Loading + Attention Hook Injection
LLaVA-1.5-7B loaded in 4-bit quantization (~5 GB VRAM instead of 14 GB).
A forward hook captures attention weights at layer k for use by compression strategies.

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)
print('Loading LLaVA-1.5-7B (this takes ~2-3 minutes)...')

In [ ]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

model_path = 'liuhaotian/llava-v1.5-7b'
model_name = get_model_name_from_path(model_path)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=model_name,
    load_4bit=True,
)
model.eval()
print('Model loaded successfully!')
print(f'LLM layers: {model.model.config.num_hidden_layers}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
import torch

# Global state
_captured_attn = {}   # {layer_idx: attn_weight tensor}
_hook_handles  = []
N_VISUAL = 576        # LLaVA-1.5 always produces 576 visual tokens

def _make_attn_hook(layer_idx):
    def hook(module, input, output):
        if isinstance(output, tuple) and len(output) > 1 and output[1] is not None:
            _captured_attn[layer_idx] = output[1].detach()
    return hook

def register_hooks(model, layers=(2, 4, 8, 16)):
    global _hook_handles
    for h in _hook_handles:
        h.remove()
    _hook_handles = []
    for idx, layer in enumerate(model.model.layers):
        if idx in layers:
            h = layer.self_attn.register_forward_hook(_make_attn_hook(idx))
            _hook_handles.append(h)
    print(f'Attention hooks registered on layers: {list(layers)}')

register_hooks(model)

## Section 3 - Compression Strategies

All strategies share the same interface:
```
compress(hidden_states, attn_weights, keep_ratio, n_visual) -> (compressed_hs, kept_idx)
```
Visual tokens are at positions `[:n_visual]`; text tokens `[n_visual:]` are never modified.

In [ ]:
import torch
import torch.nn.functional as F
import math

# ── Strategy 1: Uniform Pruning (no-signal baseline) ────────────────────────
def uniform_prune(hidden_states, attn_weights, keep_ratio, n_visual=N_VISUAL):
    k = max(1, int(n_visual * keep_ratio))
    idx = torch.linspace(0, n_visual - 1, k).long().to(hidden_states.device)
    visual_hs = hidden_states[:, :n_visual]
    text_hs   = hidden_states[:, n_visual:]
    pruned    = visual_hs[:, idx, :]
    return torch.cat([pruned, text_hs], dim=1), idx


# ── Strategy 2: Attention-guided Pruning (CORE METHOD - FastV extended) ──────
def attention_prune(hidden_states, attn_weights, keep_ratio, n_visual=N_VISUAL):
    k = max(1, int(n_visual * keep_ratio))
    # Mean attention text->visual: [B, heads, T_text, n_visual]
    text_to_visual = attn_weights[:, :, n_visual:, :n_visual]
    importance     = text_to_visual.mean(dim=(1, 2))  # [B, n_visual]
    topk_idx       = importance[0].topk(k).indices.sort().values
    visual_hs      = hidden_states[:, :n_visual]
    text_hs        = hidden_states[:, n_visual:]
    pruned         = visual_hs[:, topk_idx, :]
    return torch.cat([pruned, text_hs], dim=1), topk_idx


# ── Strategy 3: Token Merging - ToMe style (Bolya et al., ICLR 2023) ────────
def token_merge(hidden_states, attn_weights, keep_ratio, n_visual=N_VISUAL):
    k       = max(1, int(n_visual * keep_ratio))
    n_merge = n_visual - k
    visual_hs = hidden_states[:, :n_visual]
    text_hs   = hidden_states[:, n_visual:]
    merged    = visual_hs[0].clone()  # [n_visual, D]
    v_norm    = F.normalize(merged, dim=-1)
    sim       = torch.mm(v_norm, v_norm.t())
    sim.fill_diagonal_(-1)
    mask = torch.ones(n_visual, dtype=torch.bool, device=hidden_states.device)
    for _ in range(n_merge):
        valid_sim = sim.clone()
        valid_sim[~mask, :] = -1
        valid_sim[:, ~mask] = -1
        flat_idx = valid_sim.argmax()
        i, j = flat_idx // n_visual, flat_idx % n_visual
        if i == j or not mask[i] or not mask[j]:
            break
        merged[i] = (merged[i] + merged[j]) / 2
        mask[j]   = False
        sim[j, :] = -1
        sim[:, j] = -1
    kept_idx  = mask.nonzero(as_tuple=True)[0]
    merged_hs = merged[kept_idx].unsqueeze(0)
    return torch.cat([merged_hs, text_hs], dim=1), kept_idx


# ── Strategy 4: Spatial Pooling (traditional downsampling baseline) ──────────
def spatial_pool(hidden_states, attn_weights, keep_ratio, n_visual=N_VISUAL):
    # LLaVA-1.5: 576 tokens = 24x24 spatial grid
    k         = max(1, int(n_visual * keep_ratio))
    grid_size = int(math.sqrt(n_visual))  # 24
    target_h  = max(1, int(math.sqrt(k)))
    target_w  = target_h
    visual_hs = hidden_states[:, :n_visual]
    text_hs   = hidden_states[:, n_visual:]
    B, _, D   = visual_hs.shape
    grid      = visual_hs.permute(0, 2, 1).reshape(B, D, grid_size, grid_size)
    pooled    = F.adaptive_avg_pool2d(grid, (target_h, target_w))
    pooled    = pooled.reshape(B, D, -1).permute(0, 2, 1)
    kept_idx  = torch.arange(target_h * target_w, device=hidden_states.device)
    return torch.cat([pooled, text_hs], dim=1), kept_idx


STRATEGIES = {
    'baseline':  None,
    'uniform':   uniform_prune,
    'attention': attention_prune,
    'merge':     token_merge,
    'pool':      spatial_pool,
}
print('All 4 compression strategies defined:', list(STRATEGIES.keys()))

## Section 4 - Benchmark Runner
`run_experiment()` handles: timing, VRAM tracking, VQA scoring, and per-type breakdown.
Compression is applied via a temporary forward hook on the layer after pruning layer k.

In [ ]:
import time
import numpy as np
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates

def vqa_score(pred, gt):
    return 1.0 if pred.strip().lower() == gt.strip().lower() else 0.0

def run_inference_single(sample, strategy_fn, keep_ratio, prune_layer=4):
    image    = sample['image'].convert('RGB')
    question = sample['question']

    image_tensor = process_images([image], image_processor, model.config)
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    conv = conv_templates['vicuna_v1'].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + '\n' + question)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(model.device)

    _captured_attn.clear()
    tokens_used = [N_VISUAL]
    patch_handle = [None]

    if strategy_fn is not None:
        def compress_hook(module, input, output):
            if prune_layer not in _captured_attn:
                return output
            attn_w = _captured_attn[prune_layer]
            hs = output[0]
            n_text = input_ids.shape[1] - N_VISUAL
            if hs.shape[1] > (int(N_VISUAL * keep_ratio) + n_text):
                compressed, kept_idx = strategy_fn(hs, attn_w, keep_ratio)
                tokens_used[0] = kept_idx.shape[0]
                return (compressed,) + output[1:]
            return output
        attach_at = min(prune_layer + 1, len(model.model.layers) - 1)
        patch_handle[0] = model.model.layers[attach_at].register_forward_hook(compress_hook)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            image_sizes=[image.size],
            do_sample=False,
            max_new_tokens=32,
            use_cache=True,
        )

    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - t0) * 1000
    peak_vram  = torch.cuda.max_memory_allocated() / 1e9

    if patch_handle[0] is not None:
        patch_handle[0].remove()

    pred = tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    return pred, latency_ms, peak_vram, tokens_used[0]

In [ ]:
def run_experiment(strategy_name, keep_ratio, samples, prune_layer=4, verbose=True):
    strategy_fn = STRATEGIES[strategy_name]
    if strategy_name == 'baseline':
        keep_ratio = 1.0

    scores, latencies, vrams, token_counts = [], [], [], []
    by_type = {'yes/no': [], 'number': [], 'other': []}

    for i, sample in enumerate(samples):
        pred, lat, vram, n_tok = run_inference_single(
            sample, strategy_fn, keep_ratio, prune_layer
        )
        acc = vqa_score(pred, sample['gt_answer'])
        scores.append(acc)
        latencies.append(lat)
        vrams.append(vram)
        token_counts.append(n_tok)
        atype = sample.get('answer_type', 'other')
        if atype in by_type:
            by_type[atype].append(acc)
        if verbose and (i + 1) % 100 == 0:
            print(f'  [{i+1}/{len(samples)}] acc={np.mean(scores):.3f} '
                  f'lat={np.mean(latencies):.0f}ms vram={np.mean(vrams):.2f}GB')

    result = {
        'strategy':     strategy_name,
        'keep_ratio':   keep_ratio,
        'prune_layer':  prune_layer,
        'accuracy':     np.mean(scores),
        'latency_ms':   np.mean(latencies),
        'peak_vram_gb': np.mean(vrams),
        'tokens_used':  np.mean(token_counts),
        'acc_yesno':    np.mean(by_type['yes/no'])  if by_type['yes/no']  else None,
        'acc_number':   np.mean(by_type['number'])  if by_type['number']  else None,
        'acc_other':    np.mean(by_type['other'])   if by_type['other']   else None,
        'n_samples':    len(samples),
    }
    if verbose:
        print(f'\n[DONE] {strategy_name} @ {keep_ratio:.0%} | '
              f'acc={result["accuracy"]:.3f} | '
              f'lat={result["latency_ms"]:.0f}ms | '
              f'vram={result["peak_vram_gb"]:.2f}GB | '
              f'tokens={result["tokens_used"]:.0f}')
    return result

print('Experiment runner defined.')

## Section 5 - Run All Experiments

**Estimated time on A100: ~2.5-3 hours total**

Results auto-save to Google Drive after each experiment.
If Colab disconnects, re-run this cell and it will skip completed experiments.

In [ ]:
import pandas as pd
import os

RESULTS_PATH = os.path.join(SAVE_DIR, 'results.csv')

if os.path.exists(RESULTS_PATH):
    results_df = pd.read_csv(RESULTS_PATH)
    results = results_df.to_dict('records')
    print(f'Resumed: {len(results)} existing results loaded from Drive.')
else:
    results = []
    print('Starting fresh.')

def already_done(strategy, keep_ratio, prune_layer=4):
    for r in results:
        if (r['strategy'] == strategy and
            abs(r['keep_ratio'] - keep_ratio) < 0.01 and
            r['prune_layer'] == prune_layer):
            return True
    return False

In [ ]:
# ── MAIN EXPERIMENT MATRIX ───────────────────────────────────────────────────
# Exp-0: Baseline
# Exp-1: Uniform Pruning      @ 75%, 50%, 25%
# Exp-2: Attention Pruning    @ 75%, 50%, 25%  <-- CORE METHOD
# Exp-3: Token Merging        @ 75%, 50%, 25%
# Exp-4: Spatial Pooling      @ 75%, 50%, 25%

RATIOS = [0.75, 0.50, 0.25]
STRATS = ['baseline', 'uniform', 'attention', 'merge', 'pool']

for strategy in STRATS:
    ratios = [1.0] if strategy == 'baseline' else RATIOS
    for ratio in ratios:
        if already_done(strategy, ratio):
            print(f'  [skip] {strategy} @ {ratio:.0%} already done')
            continue
        print(f'\n[RUN] {strategy} @ {ratio:.0%} ...')
        r = run_experiment(strategy, ratio, samples, prune_layer=4)
        results.append(r)
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print('\nMain experiment matrix complete!')

In [ ]:
# ── EXP-5: ABLATION - Pruning Layer Position ─────────────────────────────────
print('Running ablation: prune layer k position (attention @ 50%)...')

for k in [2, 8, 16]:
    if already_done('attention', 0.50, prune_layer=k):
        print(f'  [skip] attention @ 50% layer={k}')
        continue
    print(f'\n[RUN] Attention pruning @ 50%, layer k={k}...')
    r = run_experiment('attention', 0.50, samples, prune_layer=k)
    r['prune_layer'] = k
    results.append(r)
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

results_df = pd.DataFrame(results)
print('\nAll experiments complete!')
print(results_df[['strategy','keep_ratio','prune_layer',
                   'accuracy','latency_ms','peak_vram_gb','tokens_used']])

## Section 6 - Results & Visualization
Four figures:
1. Accuracy vs Latency Pareto curve
2. Peak VRAM vs retention ratio
3. Token budget vs accuracy scatter
4. Ablation: pruning layer position

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

results_df = pd.read_csv(RESULTS_PATH)
main_df    = results_df[results_df['prune_layer'] == 4].copy()

COLORS  = {'baseline':'#2d3436','uniform':'#e17055',
           'attention':'#0984e3','merge':'#00b894','pool':'#6c5ce7'}
MARKERS = {'baseline':'D','uniform':'s','attention':'o','merge':'^','pool':'P'}
LABELS  = {'baseline':'Baseline (no compression)','uniform':'Uniform Pruning',
           'attention':'Attention-guided Pruning',
           'merge':'Token Merging (ToMe)','pool':'Spatial Pooling'}

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LLaVA-1.5-7B Visual Token Compression - Results',
             fontsize=15, fontweight='bold')

# Plot 1: Pareto Curve
ax = axes[0, 0]
for strat, grp in main_df.groupby('strategy'):
    grp = grp.sort_values('keep_ratio', ascending=False)
    ax.plot(grp['latency_ms'], grp['accuracy'],
            color=COLORS[strat], marker=MARKERS[strat],
            linewidth=2, markersize=8, label=LABELS[strat])
    if strat != 'baseline':
        for _, row in grp.iterrows():
            ax.annotate(f"{row['keep_ratio']:.0%}",
                        (row['latency_ms'], row['accuracy']),
                        textcoords='offset points', xytext=(5, 3), fontsize=7.5)
ax.set_xlabel('Average Latency (ms/sample)')
ax.set_ylabel('VQA Accuracy')
ax.set_title('Accuracy vs Latency (Pareto Curve)')
ax.legend(fontsize=8)

# Plot 2: VRAM
ax = axes[0, 1]
for strat, grp in main_df.groupby('strategy'):
    grp = grp.sort_values('keep_ratio')
    ax.plot(grp['keep_ratio'] * 100, grp['peak_vram_gb'],
            color=COLORS[strat], marker=MARKERS[strat],
            linewidth=2, markersize=8, label=LABELS[strat])
ax.set_xlabel('Visual Token Retention (%)')
ax.set_ylabel('Peak VRAM (GB)')
ax.set_title('Peak GPU Memory vs Retention Ratio')
ax.invert_xaxis()
ax.legend(fontsize=8)

# Plot 3: Token Budget vs Accuracy
ax = axes[1, 0]
for strat, grp in main_df.groupby('strategy'):
    ax.scatter(grp['tokens_used'], grp['accuracy'],
               color=COLORS[strat], marker=MARKERS[strat],
               s=100, label=LABELS[strat], zorder=3)
ax.set_xlabel('Visual Tokens Used')
ax.set_ylabel('VQA Accuracy')
ax.set_title('Token Budget vs Accuracy')
ax.legend(fontsize=8)

# Plot 4: Ablation
ax = axes[1, 1]
abl_df = results_df[
    (results_df['strategy'] == 'attention') &
    (results_df['keep_ratio'].between(0.49, 0.51))
].sort_values('prune_layer')
ax.bar(abl_df['prune_layer'].astype(str), abl_df['accuracy'],
       color='#0984e3', edgecolor='white', linewidth=1.5)
ax2 = ax.twinx()
ax2.plot(abl_df['prune_layer'].astype(str), abl_df['latency_ms'],
         color='#e17055', marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Pruning Layer k')
ax.set_ylabel('VQA Accuracy', color='#0984e3')
ax2.set_ylabel('Latency (ms)', color='#e17055')
ax.set_title('Ablation: Pruning Layer Position (50% retention)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'results_main.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Summary table with % changes vs baseline
baseline_acc = main_df[main_df['strategy'] == 'baseline']['accuracy'].values[0]
baseline_lat = main_df[main_df['strategy'] == 'baseline']['latency_ms'].values[0]

summary = main_df.copy()
summary['acc_drop_%']     = ((baseline_acc - summary['accuracy']) / baseline_acc * 100).round(2)
summary['latency_gain_%'] = ((baseline_lat - summary['latency_ms']) / baseline_lat * 100).round(2)

cols = ['strategy','keep_ratio','accuracy','acc_drop_%',
        'latency_ms','latency_gain_%','peak_vram_gb','tokens_used']
print('=' * 90)
print('FULL RESULTS TABLE')
print('=' * 90)
print(summary[cols].sort_values(['strategy','keep_ratio']).to_string(index=False))

## Section 7 - Fine-grained Analysis by Question Type

Breakdown: **Yes/No** | **Number** | **Open-ended**

Hypothesis: number/counting questions are more sensitive to visual token compression
than binary yes/no questions, because they require precise spatial reasoning.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

half_df   = main_df[main_df['keep_ratio'].between(0.49, 0.51)].copy()
all_strats = list(half_df['strategy'].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fine-grained Analysis: Accuracy by Question Type (50% token retention)',
             fontsize=13, fontweight='bold')

q_types  = ['acc_yesno', 'acc_number', 'acc_other']
q_labels = ['Yes/No', 'Number', 'Open-ended']
x = np.arange(len(q_labels))
w = 0.15

# Plot 1: Accuracy by question type per strategy
ax = axes[0]
for i, strat in enumerate(all_strats):
    row  = half_df[half_df['strategy'] == strat].iloc[0]
    vals = [row.get(qt) or 0 for qt in q_types]
    ax.bar(x + i * w, vals, w, label=LABELS[strat], color=COLORS[strat], edgecolor='white')
ax.set_xticks(x + w * (len(all_strats) - 1) / 2)
ax.set_xticklabels(q_labels)
ax.set_ylabel('Accuracy')
ax.set_title('Per-type Accuracy @ 50% Retention')
ax.legend(fontsize=7.5)
ax.set_ylim(0, 1.05)

# Plot 2: Accuracy drop vs baseline
ax = axes[1]
base_row  = main_df[main_df['strategy'] == 'baseline'].iloc[0]
base_vals = {qt: base_row.get(qt) or 0 for qt in q_types}
non_base  = [s for s in all_strats if s != 'baseline']
for i, strat in enumerate(non_base):
    row   = half_df[half_df['strategy'] == strat].iloc[0]
    drops = [((base_vals[qt] - (row.get(qt) or 0)) / max(base_vals[qt], 1e-6)) * 100
             for qt in q_types]
    ax.bar(x + i * w, drops, w, label=LABELS[strat], color=COLORS[strat], edgecolor='white')
ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(q_labels)
ax.set_ylabel('Accuracy Drop vs Baseline (%)')
ax.set_title('Sensitivity by Question Type @ 50% Retention')
ax.legend(fontsize=7.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'results_finegrained.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print key findings narrative
print('KEY FINDINGS: Question Type Sensitivity')
print('=' * 60)
base_row = main_df[main_df['strategy'] == 'baseline'].iloc[0]
for strat in ['uniform', 'attention', 'merge', 'pool']:
    rows = main_df[(main_df['strategy'] == strat) &
                   (main_df['keep_ratio'].between(0.49, 0.51))]
    if rows.empty:
        continue
    row = rows.iloc[0]
    print(f'\n{LABELS[strat]} @ 50%:')
    for qt, ql in zip(q_types, q_labels):
        base_v  = base_row.get(qt) or 0
        strat_v = row.get(qt) or 0
        drop    = (base_v - strat_v) / max(base_v, 1e-6) * 100
        print(f'  {ql:12s}: {strat_v:.3f}  (drop: {drop:+.1f}%)')

## Section 8 - Interactive Demo
Upload any image, type a question, pick a strategy and retention ratio.
Output: model answer + attention heatmap showing retained vs pruned tokens.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import io

upload_btn   = widgets.FileUpload(accept='image/*', multiple=False)
question_box = widgets.Text(
    value='What is in this image?',
    description='Question:',
    layout=widgets.Layout(width='500px')
)
strategy_dd  = widgets.Dropdown(
    options=list(STRATEGIES.keys()), value='attention', description='Strategy:'
)
ratio_slider = widgets.FloatSlider(
    value=0.5, min=0.25, max=1.0, step=0.25,
    description='Keep %:', readout_format='.0%'
)
run_btn = widgets.Button(description='Run', button_style='primary')
out     = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        if not upload_btn.value:
            print('Please upload an image first.')
            return
        img_bytes = list(upload_btn.value.values())[0]['content']
        image     = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        sample    = {'image': image, 'question': question_box.value,
                     'gt_answer': '', 'answer_type': 'other'}
        strat_name = strategy_dd.value
        ratio      = ratio_slider.value
        fn         = STRATEGIES[strat_name]
        print(f'Running {strat_name} @ {ratio:.0%}...')
        pred, lat, vram, n_tok = run_inference_single(sample, fn, ratio)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(image)
        axes[0].set_title('Input Image')
        axes[0].axis('off')

        if 4 in _captured_attn and strat_name != 'baseline':
            attn_w = _captured_attn[4]
            tov    = attn_w[0, :, N_VISUAL:, :N_VISUAL].mean(dim=(0, 1))
            tov    = tov.float().cpu().numpy()
            tov    = (tov - tov.min()) / (tov.max() - tov.min() + 1e-8)
            hm     = tov.reshape(24, 24)
            hm_up  = np.array(Image.fromarray(
                (hm * 255).astype(np.uint8)).resize(image.size, Image.BILINEAR)
            ) / 255.0
            axes[1].imshow(image, alpha=0.4)
            axes[1].imshow(hm_up, cmap='RdYlBu_r', alpha=0.6, vmin=0, vmax=1)
            axes[1].set_title('Attention Heatmap (warm=retained, cool=pruned)')
        else:
            axes[1].imshow(image)
            axes[1].set_title('Baseline (no compression)')
        axes[1].axis('off')

        plt.suptitle(
            f'Strategy: {LABELS[strat_name]} | Retention: {ratio:.0%}\n'
            f'Answer: "{pred}" | Latency: {lat:.0f}ms | '
            f'VRAM: {vram:.2f}GB | Tokens used: {n_tok}',
            fontsize=11
        )
        plt.tight_layout()
        plt.show()

run_btn.on_click(on_run)
display(widgets.VBox([
    widgets.HTML('<h3>HPML Demo - Visual Token Compression</h3>'),
    upload_btn, question_box,
    widgets.HBox([strategy_dd, ratio_slider]),
    run_btn, out
]))